# Apprentissage Automatique — Exercices Pratiques
### Prediction des defauts de paiement de prets
---

## Exercice 1 : Definition du probleme et collecte de donnees

### Enonce du probleme
L'objectif est de construire un modele de machine learning capable de **predire si un emprunteur sera en defaut de paiement** sur un pret.

Ce probleme est une **classification binaire** :
- `1` : L'emprunteur sera en defaut de paiement
- `0` : L'emprunteur remboursera correctement


In [ ]:
# EXERCICE 1 - Plan de collecte de donnees

data_plan = {
    'Informations personnelles': ['Age', 'Situation familiale', 'Niveau education', 'Situation professionnelle'],
    'Situation financiere': ['Revenu mensuel net', 'Dettes existantes', 'Ratio dette/revenu (DTI)', 'Epargne et actifs'],
    'Historique de credit': ['Score de credit', 'Nombre de retards passes', 'Nombre de prets anterieurs', 'Defauts passes'],
    'Caracteristiques du pret': ['Montant demande', 'Duree du pret (mois)', "Taux d'interet", 'Objet du pret'],
    'Sources de donnees': ['Dossiers internes institutions financieres', 'Agences de credit (Experian, Equifax)', 'Datasets publics Kaggle', 'Declarations fiscales'],
}

for categorie, elements in data_plan.items():
    print(f'\n{categorie}:')
    for e in elements:
        print(f'   - {e}')


---
## Exercice 2 : Selection des caracteristiques

| Caracteristique | Justification |
|---|---|
| Score de credit | Indicateur synthetique du risque |
| Ratio dette/revenu | Mesure la capacite de remboursement |
| Historique de paiement | Comportement passe = meilleur predicteur |
| Montant du pret | Montant eleve = risque plus grand |
| Revenu mensuel | Capacite a honorer les echeances |
| Nombre de retards | Signal direct de comportement a risque |


In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn --quiet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier

# Dataset synthetique
np.random.seed(42)
n = 1000
df = pd.DataFrame({
    'score_credit': np.random.randint(300, 850, n),
    'ratio_dette_revenu': np.random.uniform(0.1, 0.9, n),
    'montant_pret': np.random.randint(1000, 50000, n),
    'duree_pret_mois': np.random.choice([12, 24, 36, 48, 60], n),
    'revenu_mensuel': np.random.randint(1000, 10000, n),
    'nb_retards': np.random.randint(0, 10, n),
    'anciennete_emploi': np.random.randint(0, 30, n),
})
df['defaut'] = ((df['score_credit'] < 550) | (df['ratio_dette_revenu'] > 0.6) | (df['nb_retards'] > 5)).astype(int)

X = df.drop('defaut', axis=1)
y = df['defaut']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

imp = pd.DataFrame({'Feature': X.columns, 'Importance': rf.feature_importances_}).sort_values('Importance')

plt.figure(figsize=(9, 5))
plt.barh(imp['Feature'], imp['Importance'], color='steelblue')
plt.xlabel('Importance (Gini)')
plt.title('Importance des caracteristiques - Random Forest')
plt.tight_layout()
plt.show()


---
## Exercice 3 : Entrainement, evaluation et optimisation

**Modeles choisis :**
1. Regression Logistique (baseline interpretable)
2. Random Forest (robuste sur donnees tabulaires)
3. XGBoost (tres performant en classification financiere)

**Metriques cles :** AUC-ROC, Precision, Rappel, F1-Score


In [ ]:
!pip install xgboost --quiet


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

models = {
    'Regression Logistique': (LogisticRegression(random_state=42, max_iter=1000), True),
    'Random Forest': (RandomForestClassifier(n_estimators=100, random_state=42), False),
    'XGBoost': (XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0), False),
}

results = {}
for name, (model, scaled) in models.items():
    Xtr = X_train_sc if scaled else X_train
    Xte = X_test_sc if scaled else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]
    results[name] = {'y_pred': y_pred, 'y_prob': y_prob, 'auc': roc_auc_score(y_test, y_prob)}
    print(f'\n{name} - AUC: {results[name]["auc"]:.4f}')
    print(classification_report(y_test, y_pred, target_names=['Non-defaut', 'Defaut']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#e74c3c', '#2ecc71', '#3498db']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", color=color, lw=2)
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_title('Courbes ROC')
axes[0].set_xlabel('Taux faux positifs')
axes[0].set_ylabel('Taux vrais positifs')
axes[0].legend()
axes[0].grid(alpha=0.3)

best = max(results, key=lambda k: results[k]['auc'])
cm = confusion_matrix(y_test, results[best]['y_pred'])
ConfusionMatrixDisplay(cm, display_labels=['Non-defaut','Defaut']).plot(ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title(f'Matrice de confusion - {best}')

plt.tight_layout()
plt.show()
print(f'Meilleur modele : {best} (AUC = {results[best]["auc"]:.4f})')


---
## Exercice 4 : Type d'apprentissage par scenario

| Scenario | Type | Justification |
|---|---|---|
| Prevision cours boursiers | Supervise (Regression) | Donnees historiques etiquetees, valeur continue a predire |
| Organiser une bibliotheque | Non supervise (Clustering) | Pas de labels, groupement par similarite |
| Robot dans un labyrinthe | Apprentissage par renforcement | Essais/erreurs avec recompenses |


In [ ]:
# SCENARIO 2 : Clustering de livres (bibliotheque)
from sklearn.cluster import KMeans

np.random.seed(42)
livres = np.vstack([
    np.random.randn(50, 2) + [0, 4],
    np.random.randn(50, 2) + [4, 0],
    np.random.randn(50, 2) + [-4, 0],
    np.random.randn(50, 2) + [0, -4],
])
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(livres)

genres = {0: 'Sci-Fi', 1: 'Histoire', 2: 'Romance', 3: 'Policier'}
colors_c = ['#e74c3c', '#2ecc71', '#e67e22', '#3498db']
plt.figure(figsize=(7, 5))
for i in range(4):
    mask = labels == i
    plt.scatter(livres[mask,0], livres[mask,1], c=colors_c[i], label=genres[i], alpha=0.7)
plt.scatter(kmeans.cluster_centers_[:,0], kmeans.cluster_centers_[:,1], c='black', marker='X', s=200, zorder=5)
plt.title('Clustering de livres par genres (K-Means)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# SCENARIO 3 : Q-Learning dans un labyrinthe
import random

maze = [[0,0,1,0],[1,0,1,0],[0,0,0,0],[0,1,0,0]]
ROWS, COLS = 4, 4
START, GOAL = (0,0), (3,3)
ACTIONS = [(-1,0),(1,0),(0,-1),(0,1)]
Q = np.zeros((ROWS, COLS, 4))
alpha, gamma, epsilon = 0.1, 0.9, 0.3
rewards_hist = []

for ep in range(500):
    state = START
    total = 0
    for _ in range(100):
        r, c = state
        a = random.randint(0,3) if random.random() < epsilon else np.argmax(Q[r,c])
        dr, dc = ACTIONS[a]
        nr, nc = r+dr, c+dc
        ns = (nr,nc) if 0<=nr<ROWS and 0<=nc<COLS and maze[nr][nc]==0 else state
        rew = 10 if ns==GOAL else -0.1
        Q[r,c,a] += alpha*(rew + gamma*np.max(Q[ns[0],ns[1]]) - Q[r,c,a])
        state = ns
        total += rew
        if state == GOAL: break
    rewards_hist.append(total)

smooth = np.convolve(rewards_hist, np.ones(20)/20, mode='valid')
plt.figure(figsize=(9,4))
plt.plot(rewards_hist, alpha=0.3, color='steelblue', label='Par episode')
plt.plot(range(19, len(rewards_hist)), smooth, color='red', lw=2, label='Moyenne mobile 20')
plt.xlabel('Episode')
plt.ylabel('Recompense cumulee')
plt.title('Q-Learning - Courbe apprentissage robot labyrinthe')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Recompense moyenne (50 derniers episodes): {np.mean(rewards_hist[-50:]):.2f}')


---
## Exercice 5 : Strategie d'evaluation pour 3 types de modeles

### 1. Supervise (Classification)
- **Metriques** : Accuracy, Precision, Rappel, F1, AUC-ROC
- **Methodes** : Validation croisee k-fold, courbes ROC
- **Difficultes** : Desequilibre des classes, choix du seuil

### 2. Non supervise (Clustering)
- **Metriques** : Score de silhouette, methode du coude, Davies-Bouldin
- **Difficultes** : Pas de verite terrain, sensibilite au k

### 3. Apprentissage par renforcement
- **Metriques** : Recompense cumulative, convergence, equilibre exploration/exploitation
- **Difficultes** : Instabilite, definition de la fonction de recompense


In [ ]:
# EVALUATION SUPERVISE - Metriques detaillees
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

best_name = max(results, key=lambda k: results[k]['auc'])
yp = results[best_name]['y_pred']
ypr = results[best_name]['y_prob']

metriques = {
    'Accuracy': accuracy_score(y_test, yp),
    'Precision': precision_score(y_test, yp),
    'Rappel': recall_score(y_test, yp),
    'F1-Score': f1_score(y_test, yp),
    'AUC-ROC': roc_auc_score(y_test, ypr),
}

print(f'Modele : {best_name}')
print('-' * 40)
for m, v in metriques.items():
    bar = '#' * int(v * 20)
    print(f'  {m:<12}: {v:.4f}  {bar}')
print('\nNote: Dans le contexte financier, le RAPPEL est prioritaire')


In [ ]:
# EVALUATION NON SUPERVISE - Silhouette + Coude
from sklearn.metrics import silhouette_score, davies_bouldin_score

inertias, silhouettes, db_scores = [], [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(livres)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(livres, lbl))
    db_scores.append(davies_bouldin_score(livres, lbl))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(list(K_range), inertias, 'bo-')
axes[0].axvline(x=4, color='red', linestyle='--')
axes[0].set_title('Methode du Coude')
axes[0].set_xlabel('k')
axes[0].grid(alpha=0.3)

axes[1].plot(list(K_range), silhouettes, 'go-')
axes[1].axvline(x=4, color='red', linestyle='--')
axes[1].set_title('Score de Silhouette (plus haut = meilleur)')
axes[1].set_xlabel('k')
axes[1].grid(alpha=0.3)

axes[2].plot(list(K_range), db_scores, 'ro-')
axes[2].axvline(x=4, color='blue', linestyle='--')
axes[2].set_title('Davies-Bouldin (plus bas = meilleur)')
axes[2].set_xlabel('k')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

km4 = KMeans(n_clusters=4, random_state=42, n_init=10).fit(livres)
lbl4 = km4.labels_
print(f'k=4 -> Silhouette: {silhouette_score(livres, lbl4):.4f}')
print(f'k=4 -> Davies-Bouldin: {davies_bouldin_score(livres, lbl4):.4f}')


In [ ]:
# EVALUATION APPRENTISSAGE PAR RENFORCEMENT
window_sizes = [10, 20, 50]
colors_rl = ['#e74c3c', '#2ecc71', '#3498db']

plt.figure(figsize=(9, 4))
plt.plot(rewards_hist, alpha=0.2, color='gray', label='Par episode')
for w, col in zip(window_sizes, colors_rl):
    sm = np.convolve(rewards_hist, np.ones(w)/w, mode='valid')
    plt.plot(range(w-1, len(rewards_hist)), sm, color=col, lw=2, label=f'Fenetre {w}')
plt.title('Convergence RL - Recompense cumulee')
plt.xlabel('Episode')
plt.ylabel('Recompense')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('METRIQUES RL :')
print(f'  Recompense moyenne (100 derniers): {np.mean(rewards_hist[-100:]):.3f}')
print(f'  Recompense max atteinte          : {max(rewards_hist):.3f}')
print('\nDifficultes RL :')
print('  - Instabilite entrainement')
print('  - Definir une bonne fonction de recompense')
print('  - Risque de sur-specialisation')


---
## Conclusion

| Exercice | Statut |
|---|---|
| Ex.1 : Enonce + collecte de donnees | OK |
| Ex.2 : Selection des features | OK |
| Ex.3 : Entrainement et evaluation | OK |
| Ex.4 : Type d'apprentissage par scenario | OK |
| Ex.5 : Strategie evaluation 3 types | OK |

*Notebook realise dans le cadre du cours d'Apprentissage Automatique - DI Learning*
